# 01 — Manifest, split S1, pilot selection (Day 1 Step 2)

Runs exactly the same code as `uv run python -m bdsl.cli manifest | split | pilot-select` (see `CLAUDE.md` → COMMANDS).

- **Run it with** the project's `.venv` kernel. Paths come from `configs/` (and `BDSL_*` env vars), never from this notebook.
- **Data safety:** videos and the sheet are only read; size + mtime of every video and the sheet SHA-256 are compared before and after.
- **Long job:** the manifest hashes all 5,010 videos (~71 GB, ~2–3 min on the desktop). It resumes from `work/manifests/manifest_v1.partial.jsonl`, so a re-run reuses earlier hashes. For long runs a terminal is safer than a notebook.
- **Approval gates:** copying the pilot clips (`pilot-copy --confirm`) happens only after the owner has approved the 50-clip list (ADR-0017, ADR-0022).

In [ ]:
from bdsl.cli import main
from bdsl.config import load_config

cfg = load_config(data="full")
print("env:", cfg["_meta"]["env"], "| data root:", cfg["paths"]["data_root"], "| work dir:", cfg["paths"]["work_dir"])

In [ ]:
# Manifest v1: SHA-256 + OpenCV header data + norm_v1 sentences + leakage groups (read-only on sources).
print("exit code:", main(["manifest", "--data", "full"]))

In [ ]:
# Split S1 (group-disjoint, deterministic) + automated leakage checks + balance report (ADR-0018).
print("exit code:", main(["split"]))

In [ ]:
# pilot50_v1 selection: writes a LIST only (work/pilot/), copies nothing.
print("exit code:", main(["pilot-select"]))

**Copying the pilot clips** needs the owner's approval of the list printed above. Only then uncomment and run the next cell. It copies with a temp file + rename, verifies every SHA-256 and never touches the originals.

In [ ]:
# print("exit code:", main(["pilot-copy", "--confirm"]))  # owner-approved list only

In [ ]:
# Summary from the saved meta files.
import json
from pathlib import Path

p = cfg["paths"]
manifest_meta = json.loads((Path(p["manifests_dir"]) / "manifest_v1.meta.json").read_text(encoding="utf-8"))
split_meta = json.loads((Path(p["splits_dir"]) / "S1_v0.meta.json").read_text(encoding="utf-8"))
pilot_meta = json.loads((Path(p["pilot_work_dir"]) / "pilot50_v1_selection.meta.json").read_text(encoding="utf-8"))
c = manifest_meta["counts"]
print("manifest rows:", c["sheet_rows"], "| unique files:", c["unique_sha256"], "| sentences:", c["unique_sentence_norm"],
      "| groups:", c["groups"], "| label conflicts:", c["label_conflicts"], "| sources unchanged:", manifest_meta["sources_unchanged"])
print("split:", split_meta["split"], "| leakage problems:", split_meta["leakage_problems"] or "none", "| groups:", split_meta["balance"]["groups"])
r = pilot_meta["report"]
print("pilot ok:", r["ok"], "| counts:", r["counts"], "| held-out clips using only seen words:", r["heldout_all_words_seen"], "/", r["heldout_total"])